# Public Speaking dimension — end-to-end demo (Module 9)

**Runs on Google Colab GPU only** (Runtime → Change runtime type → **T4 GPU**).
Drive it from VS Code via the Google Colab extension, then **Run all**.

| Real (this notebook computes it) | Stubbed (owned by another module) |
|---|---|
| eye_contact, posture (YOLOv8-pose) | slide_structure → Module 4 |
| speech_pace, voice_stability (Librosa) | audience_engagement → Module 8 |
| opening_closing_impact (Qwen2.5-3B LLM) | |

No API keys — all intelligence is local/open-weight.


## Step 1 — Install deps + load the local LLM

In [ ]:
# Colab-only installs (multi-GB; never run locally)
%pip install -q ultralytics==8.3.0 openai-whisper==20231117 librosa==0.10.2.post1 \
    ffmpeg-python==0.2.0 transformers==4.44.2 accelerate==0.34.2 soundfile==0.12.1 requests==2.32.3
print("deps installed")

In [ ]:
# Make the project package importable (walk up to find utils/taxonomy.py).
import os, sys
def find_root(start):
    d = os.path.abspath(start)
    for _ in range(8):
        if os.path.exists(os.path.join(d, "utils", "taxonomy.py")):
            return d
        d = os.path.dirname(d)
    return None
ROOT = find_root(os.getcwd())
if ROOT is None:
    raise RuntimeError("Project root not found. Upload/clone the personalised-lms-service "
                       "folder so utils/taxonomy.py is reachable, then re-run.")
sys.path.insert(0, ROOT)
print("project root:", ROOT)

In [ ]:
# Load Qwen2.5-3B-Instruct once; wrap as a plain llm_generate(prompt)->str callable.
# (No global state leaks into the scorers — we PASS this function in.)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # set to "Qwen/Qwen2.5-1.5B-Instruct" if VRAM is tight
tok = AutoTokenizer.from_pretrained(MODEL_ID)
mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

def llm_generate(prompt: str) -> str:
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    out = mdl.generate(**inputs, max_new_tokens=512, do_sample=False)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("LLM ready:", MODEL_ID)

## Step 2 — Get a clip → frames → YOLOv8-pose → eye_contact + posture

Tries a license-clean download; on **any** failure falls back to a synthetic clip
(and synthetic demo pose/transcript) so the notebook always runs end-to-end.

In [ ]:
import os, subprocess, hashlib, glob, requests

# Swap SAMPLE_URL for your own clip, or upload an .mp4 and set VIDEO directly.
SAMPLE_URL = "https://upload.wikimedia.org/wikipedia/commons/transcoded/9/9b/" \
             "Sample_speech.ogv/Sample_speech.ogv.360p.webm"  # public-domain example
EXPECTED_SHA256 = ""   # paste the hash after first download to enforce integrity ("" = skip+warn)
VIDEO = "sample_clip.webm"
USED_SYNTHETIC = False

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

try:
    r = requests.get(SAMPLE_URL, timeout=20)
    r.raise_for_status()
    with open(VIDEO, "wb") as f:
        f.write(r.content)
    digest = _sha256(VIDEO)
    if EXPECTED_SHA256 and digest != EXPECTED_SHA256:
        raise ValueError(f"sha256 mismatch: {digest}")
    if not EXPECTED_SHA256:
        print("WARNING: EXPECTED_SHA256 not set — integrity not enforced. Got:", digest)
    print("downloaded sample clip")
except Exception as e:
    print("download failed (", e, ") -> synthetic fallback")
    USED_SYNTHETIC = True
    VIDEO = "sample_clip.mp4"
    # 6s test pattern + sine tone, fully offline. subprocess list args = no shell injection.
    subprocess.run(["ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc=size=640x480:rate=2:duration=6",
                    "-f", "lavfi", "-i", "sine=frequency=220:duration=6", "-shortest",
                    VIDEO], check=True, capture_output=True)


In [ ]:
# Extract frames at 2 fps and run pose estimation.
from ultralytics import YOLO
import numpy as np

os.makedirs("frames", exist_ok=True)
for p in glob.glob("frames/*.jpg"):
    os.remove(p)
subprocess.run(["ffmpeg", "-y", "-i", VIDEO, "-vf", "fps=2", "frames/f_%04d.jpg"],
               check=True, capture_output=True)
frame_paths = sorted(glob.glob("frames/*.jpg"))

COCO = ["nose","left_eye","right_eye","left_ear","right_ear","left_shoulder","right_shoulder",
        "left_elbow","right_elbow","left_wrist","right_wrist","left_hip","right_hip",
        "left_knee","right_knee","left_ankle","right_ankle"]

pose_model = YOLO("yolov8n-pose.pt")
pose_frames = []
for res in pose_model(frame_paths, verbose=False):
    if res.keypoints is None or res.keypoints.data.shape[0] == 0:
        pose_frames.append({"keypoints": {}})
        continue
    kp = res.keypoints.data[0].cpu().numpy()  # first person, (17,3)
    pose_frames.append({"keypoints": {COCO[i]: (float(kp[i,0]), float(kp[i,1]), float(kp[i,2]))
                                      for i in range(len(COCO))}})
print(f"{len(pose_frames)} frames; persons detected in "
      f"{sum(1 for f in pose_frames if f['keypoints'])}")

In [ ]:
# Synthetic clip has no person; inject clearly-labelled demo data so the rest is rich.
if USED_SYNTHETIC or not any(f["keypoints"] for f in pose_frames):
    print("NOTE: using SYNTHETIC demo pose data (no real person detected).")
    pose_frames = [{"keypoints": {
        "nose": (50, 30, 0.9), "left_eye": (40, 25, 0.9), "right_eye": (60, 25, 0.9),
        "left_shoulder": (38, 100, 0.9), "right_shoulder": (62, 104, 0.9),
        "left_hip": (42, 200, 0.9), "right_hip": (58, 200, 0.9)}}] * 6
    # turn a couple of frames away to make eye_contact < perfect
    pose_frames[0] = {"keypoints": {"nose": (95, 30, 0.9), "left_eye": (40, 25, 0.9),
                                    "right_eye": (60, 25, 0.9)}}

from dimensions.public_speaking.scorer_ps import score_eye_contact, score_posture
s_eye = score_eye_contact(pose_frames)
s_post = score_posture(pose_frames)
print(s_eye)
print(s_post)

## Step 3 — Audio → Librosa → speech_pace + voice_stability

In [ ]:
import librosa
import numpy as np

subprocess.run(["ffmpeg", "-y", "-i", VIDEO, "-ac", "1", "-ar", "16000", "audio.wav"],
               check=True, capture_output=True)
y, sr = librosa.load("audio.wav", sr=16000)
duration_s = float(librosa.get_duration(y=y, sr=sr))
f0, _, _ = librosa.pyin(y, fmin=80, fmax=400, sr=sr)
f0 = np.nan_to_num(f0)
rms = librosa.feature.rms(y=y)[0]
print(f"duration={duration_s:.1f}s  voiced_frames={(f0>0).sum()}")

In [ ]:
import whisper
asr = whisper.load_model("base")
transcript = asr.transcribe("audio.wav").get("text", "").strip()

if USED_SYNTHETIC or len(transcript.split()) < 12:
    print("NOTE: using SYNTHETIC demo transcript (synthetic/short audio).")
    transcript = ("Good morning everyone and thank you for being here today. "
                  "I want to share three ideas that changed how I work. "
                  "First, focus beats hustle. Second, feedback is a gift. "
                  "Third, ship small and often. Remember this: progress compounds, "
                  "so start today and let your future self thank you.")
word_count = len(transcript.split())

from dimensions.public_speaking.scorer_ps import score_speech_pace, score_voice_stability
s_pace = score_speech_pace(word_count, duration_s if duration_s > 0 else 30.0)
s_voice = score_voice_stability(f0, rms)
print(s_pace)
print(s_voice)

## Step 4 — Whisper transcript → LLM → opening_closing_impact

In [ ]:
from dimensions.public_speaking.scorer_ps import score_opening_closing
s_open = score_opening_closing(transcript, llm_generate)
print(s_open)
print("justification:", s_open.detail.get("justification"))

## Step 5 — Stubbed sub-skills (owned by Modules 4 & 8) — clearly MOCK

In [ ]:
from dimensions.public_speaking.stubs.module4_stub import stub_slide_structure
from dimensions.public_speaking.stubs.module8_stub import stub_audience_engagement

STUDENT_ID = "stu-2026-001"
s_slide = stub_slide_structure(STUDENT_ID)
s_aud = stub_audience_engagement(STUDENT_ID)
print("MOCK", s_slide)
print("MOCK", s_aud)

all_scores = [s_eye, s_post, s_pace, s_voice, s_open, s_slide, s_aud]
for s in all_scores:
    print(f"  {s.skill:24s} {s.score:5.2f}  [{s.source.value}]")

## Step 6 — Gap detection over the 7 scores

In [ ]:
from services.gap_detector import detect_gaps
from dimensions.public_speaking.taxonomy_ps import DEMAND_BOOSTS

gaps = detect_gaps(all_scores, DEMAND_BOOSTS)
for g in gaps:
    print(f"  {g.skill:24s} score={g.score:.2f} severity={g.gap_severity:.2f} "
          f"priority={g.priority_score:.2f}")
top5 = gaps[:5]

## Step 7 — Generate an adaptive learning path (LLM)

In [ ]:
from services.path_generator import generate_learning_path

path = generate_learning_path(
    target_role="Data Analyst",
    top_gaps=top5,
    avg_completion_hours=4,
    pace="moderate",
    preferred_resource_types=["video", "exercise", "article"],
    llm_generate=llm_generate,
)
print(f"{len(path)} steps generated")

## Step 8 — Print the path + gap-severity chart (dashboard stand-in)

In [ ]:
for i, step in enumerate(path):
    pre = step["prerequisite_step_index"]
    print(f"{i}. [{step['difficulty']}] {step['title']} ({step['resource_type']}, "
          f"{step['estimated_minutes']}m) -> {step['skill_addressed']}"
          + (f"  (after step {pre})" if pre is not None else ""))
    print(f"     reason: {step['reason']}")

In [ ]:
import matplotlib.pyplot as plt

def band_color(score):
    return "#d64545" if score < 4 else ("#e0a82e" if score < 6 else "#3d9970")  # red/amber/green

labels = [s.skill for s in all_scores]
vals = [s.score for s in all_scores]
colors = [band_color(v) for v in vals]

plt.figure(figsize=(9, 4))
plt.bar(labels, vals, color=colors)
plt.axhline(6.0, ls="--", c="grey", label="gap threshold (6.0)")
plt.ylim(0, 10); plt.ylabel("score / 10"); plt.xticks(rotation=30, ha="right")
plt.title("Public Speaking — sub-skill scores"); plt.legend(); plt.tight_layout()
plt.show()

## Step 9 — Adaptive sequencing demo

Rule: a step completed at **>85%** accelerates the next step's difficulty; a skill
scoring **<50% twice** gets a simpler prerequisite inserted. Uses fake completion events.

In [ ]:
from services.path_generator import adapt_path

fake_events = [
    {"step_index": 0, "skill": path[0]["skill_addressed"], "score_pct": 92},   # accelerate next
    {"skill": top5[0].skill, "score_pct": 40},                                  # struggle 1
    {"skill": top5[0].skill, "score_pct": 47},                                  # struggle 2 -> prereq
]
adapted = adapt_path(path, fake_events)
print(f"original {len(path)} steps -> adapted {len(adapted)} steps")
for i, step in enumerate(adapted):
    print(f"{i}. [{step['difficulty']}] {step['title']} -> {step['skill_addressed']}")

## Step 10 — Emit the gap profile to the shared analytics module (Module 5)

In [ ]:
from services.analytics_emit import emit_gap_profile_updated
payload = emit_gap_profile_updated(STUDENT_ID, gaps)
print("\nemitted event:", payload["event"], "for", payload["student_id"])